# 532 nm and 1064 nm Spectral Characterization

**Instrument:** HORIBA iHR550 with Synapse Plus detector. The iHR550 has a 550 mm focal length and f/6.4 entrance aperture. Files were acquired at the front entrance by free-space coupling, one accumulation, binning 1, and LabSpec `signal-dark` correction. A filename slit value of 0 µm means the minimum/fully closed setting; it is not interpreted as a calibrated zero-width physical aperture.

A visible mercury lamp was used to calibrate two dominant lines for each grating. This supports useful wavelength work near 532 nm, but four displayed digits are not an uncertainty estimate. At 1064 nm the visible calibration is an extrapolation because no near-IR reference line or calibration residual was recorded.

The nonparametric FWHM is calculated after robust baseline removal. No nominal spectrometer resolution is quadratically subtracted: an intrinsic linewidth requires a measured line-spread function under the same grating, slit, detector, wavelength, and acquisition conditions.

In [ ]:
from pathlib import Path
import json, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def _locate_project_root(start=None):
    """Find the extracted project folder even if Jupyter was launched one directory above it."""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for base in candidates:
        if (base / "laser_characterization.py").is_file() and (base / "outputs").is_dir():
            return base.resolve()
        try:
            children = sorted(base.glob("Laser_Characterization_Project*")) + sorted(base.glob("laser_characterization_project*"))
        except OSError:
            children = []
        for child in children:
            if child.is_dir() and (child / "laser_characterization.py").is_file() and (child / "outputs").is_dir():
                return child.resolve()
    raise FileNotFoundError(
        "Could not locate the laser-characterization project folder. Keep the notebooks, "
        "laser_characterization.py, laser_characterization_core.py, and outputs/ together."
    )

PROJECT_ROOT = _locate_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from laser_characterization import *

OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = PROJECT_ROOT / "figures"
FIGURE_DIR.mkdir(exist_ok=True)
with (PROJECT_ROOT / "experiment_metadata.json").open() as handle:
    EXPERIMENT = json.load(handle)

# Raw data are optional for viewing the complete precomputed analysis.
# If present, they enable reconstruction of raw-data plots.
try:
    DATA_ROOT = find_data_root(PROJECT_ROOT)
except FileNotFoundError:
    DATA_ROOT = None
    for candidate in [
        PROJECT_ROOT / "data" / "532 and 1064",
        PROJECT_ROOT.parent / "532 and 1064",
        PROJECT_ROOT.parent / "data" / "532 and 1064",
    ]:
        if candidate.is_dir():
            DATA_ROOT = candidate.resolve()
            break
if DATA_ROOT is None:
    warnings.warn(
        "Raw data folder not found. This is OK: all precomputed result tables are included, "
        "and raw-data-only plots will be skipped."
    )


def source_path(stored_path):
    """Resolve a portable path stored relative to the `532 and 1064` data root."""
    p = Path(str(stored_path))
    if p.is_absolute() and p.exists():
        return p
    if DATA_ROOT is None:
        return p
    # Current V4 tables store paths relative to DATA_ROOT.
    q = Path(DATA_ROOT) / p
    if q.exists():
        return q
    # Backward compatibility for older absolute paths.
    parts = p.parts
    if "532 and 1064" in parts:
        i = parts.index("532 and 1064")
        q = Path(DATA_ROOT) / Path(*parts[i+1:])
        if q.exists():
            return q
    return q

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
})
print("Project:", PROJECT_ROOT)
print("Raw data:", DATA_ROOT if DATA_ROOT else "not mounted (using included precomputed results)")


In [ ]:
spectral = pd.read_csv(OUTPUT_DIR / "spectral_metrics.csv", parse_dates=["acquired_datetime"])
repeatability = pd.read_csv(OUTPUT_DIR / "spectral_repeatability_sequences.csv")
ps_limits = pd.read_csv(OUTPUT_DIR / "spectral_picosecond_transform_limits.csv")

print(f"Spectra: {len(spectral)}")
print("Conservative high-count warnings:", int(spectral.near_saturation.sum()))

## 532 nm CW grating comparison

All four 532 nm peaks have fewer than 2.2 samples across the half-maximum width. The measured FWHM values are therefore instrument/sampling-limited upper bounds, not the intrinsic COMPASS linewidth. Center wavelengths near 532 nm are plausible and benefit from visible-region Hg calibration, but a formal absolute uncertainty cannot be reconstructed.

In [ ]:
s532 = spectral[spectral.wavelength_family_nm.eq(532)].sort_values(["grating_g_mm", "acquisition_s_filename"])
display(s532[[
    "grating_g_mm", "acquisition_s_filename", "peak_wavelength_nm", "centroid_wavelength_nm",
    "fwhm_nm", "sampling_step_nm", "points_per_fwhm", "width_status",
]].round(6))

In [ ]:
if DATA_ROOT is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), constrained_layout=True)
    for _, row in s532.iterrows():
        result, arr = analyze_spectrum(source_path(row["path"]), SpectralConfig(), keep_arrays=True)
        y = arr["corrected_counts"] / max(np.max(arr["corrected_counts"]), 1)
        label = f'{int(row.grating_g_mm)} gr/mm, {row.acquisition_s_filename:g} s'
        axes[0].plot(arr["wavelength_nm"], y, marker=".", markersize=3, label=label)
        axes[1].plot(arr["wavelength_nm"], y, marker="o", markersize=3, label=label)
    axes[0].set_xlabel("Wavelength [nm]")
    axes[0].set_ylabel("Normalized corrected signal")
    axes[0].set_title("Full recorded windows")
    axes[0].legend(fontsize=8)
    axes[1].set_xlim(531.5, 532.7)
    axes[1].set_xlabel("Wavelength [nm]")
    axes[1].set_ylabel("Normalized corrected signal")
    axes[1].set_title("Peak region; markers expose coarse sampling")
    axes[1].legend(fontsize=8)
    fig.suptitle("532 nm CW grating comparison")
    fig.savefig(FIGURE_DIR / "spectral_532_gratings_v2.png", bbox_inches="tight")
    plt.show()

## 1064 nm CW slit series: direct evidence of instrumental broadening

At 1200 gr/mm, widening the nominal entrance slit from 10 to 200 µm increases the measured FWHM from about 0.0559 to 0.2246 nm. This demonstrates that the measured narrow-line width is strongly configuration dependent.

In [ ]:
cw_slit = spectral[
    spectral.wavelength_family_nm.eq(1064)
    & spectral["mode"].eq("cw")
    & spectral.grating_g_mm.eq(1200)
].sort_values("slit_um_filename")
display(cw_slit[[
    "slit_um_filename", "peak_wavelength_nm", "centroid_wavelength_nm", "fwhm_nm",
    "points_per_fwhm", "width_status",
]].round(6))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(cw_slit.slit_um_filename, cw_slit.fwhm_nm, marker="o")
ax.set_xlabel("Nominal entrance slit [µm]")
ax.set_ylabel("Measured FWHM [nm]")
ax.set_title("1064 nm CW: measured width is slit dependent")
fig.savefig(FIGURE_DIR / "spectral_1064_slit_broadening_v2.png", bbox_inches="tight")
plt.show()

## 65 ns spectra: acquisition-state and high-count warning

The 300 gr/mm, 10 µm series separates into an early lower-count sequence with FWHM around 0.52–0.58 nm and a later high-count sequence with FWHM around 2.28–2.38 nm. Four later files exceed the project's conservative 85% full-scale quality-control threshold. This threshold is not a manufacturer linearity specification; it is a warning that linearity was not established close to the exported full scale. These sequences must not be averaged together as ordinary repeats.

In [ ]:
s65 = spectral[
    spectral.wavelength_family_nm.eq(1064)
    & spectral["mode"].eq("65_ns")
    & spectral.grating_g_mm.eq(300)
    & spectral.slit_um_filename.eq(10)
].sort_values("acquired_datetime")
display(s65[[
    "acquired_datetime", "file", "peak_raw_counts", "detector_full_scale_fraction",
    "peak_wavelength_nm", "fwhm_nm", "near_saturation", "sequence_id", "width_status",
]].round(6))

In [ ]:
if DATA_ROOT is not None:
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for _, row in s65.iterrows():
        result, arr = analyze_spectrum(source_path(row["path"]), SpectralConfig(), keep_arrays=True)
        y = arr["corrected_counts"] / max(np.max(arr["corrected_counts"]), 1)
        alpha = 0.45 if row.near_saturation else 0.9
        ax.plot(arr["wavelength_nm"], y, alpha=alpha,
                label=f'{row.acquired_datetime.time()} {row.sequence_id}, {row.detector_full_scale_fraction:.1%} FS')
    ax.set_xlim(1056, 1063)
    ax.set_xlabel("Wavelength [nm]")
    ax.set_ylabel("Normalized corrected signal")
    ax.set_title("65 ns, 300 gr/mm, 10 µm: separate acquisition states")
    ax.legend(fontsize=7, ncol=2)
    fig.savefig(FIGURE_DIR / "spectral_65ns_sequences_v2.png", bbox_inches="tight")
    plt.show()

## 1064 nm operating-mode comparison

The narrow CW/65 ns peaks lie near 1059–1061 nm. That is within the GSL106A manufacturer band of 1064 ± 10 nm and is therefore physically plausible. It does not prove that the absolute calibration at 1064 nm is correct. The broadband picosecond-mode spectra peak/centroid nearer 1053–1055 nm; the 1200 gr/mm peak near 1052.89 nm lies just below the nominal lower limit, but the visible-only wavelength calibration and uncorrected system spectral response prevent a definitive source-wavelength conclusion.

In [ ]:
representative = pd.concat([
    spectral[(spectral["mode"].eq("cw")) & spectral.grating_g_mm.eq(1200) & spectral.slit_um_filename.eq(10)].head(1),
    spectral[(spectral["mode"].eq("65_ns")) & spectral.grating_g_mm.eq(1200) & spectral.slit_um_filename.eq(10)].head(1),
    spectral[(spectral["mode"].eq("picosecond")) & spectral.grating_g_mm.eq(1200) & spectral.slit_um_filename.eq(10)].head(1),
])
display(representative[[
    "mode", "peak_wavelength_nm", "centroid_wavelength_nm", "fwhm_nm",
    "points_per_fwhm", "peak_within_gsl_1064_plusminus10_nm",
    "centroid_within_gsl_1064_plusminus10_nm", "absolute_center_claim_status",
]].round(6))

In [ ]:
if DATA_ROOT is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    for _, row in representative.iterrows():
        result, arr = analyze_spectrum(source_path(row["path"]), SpectralConfig(), keep_arrays=True)
        y = arr["corrected_counts"] / max(np.max(arr["corrected_counts"]), 1)
        axes[0].plot(arr["wavelength_nm"], y, label=row["mode"])
        axes[1].plot(arr["wavelength_nm"], y, marker=".", markersize=3, label=row["mode"])
    axes[0].axvspan(1054, 1074, alpha=0.08, label="GSL106A 1064 ± 10 nm band")
    axes[0].set_xlim(1039, 1075)
    axes[0].set_xlabel("Wavelength [nm]")
    axes[0].set_ylabel("Normalized corrected signal")
    axes[0].set_title("Full 1200 gr/mm windows")
    axes[1].set_xlim(1058.8, 1061.3)
    axes[1].set_xlabel("Wavelength [nm]")
    axes[1].set_ylabel("Normalized corrected signal")
    axes[1].set_title("Narrow-line region")
    axes[0].legend(fontsize=8)
    axes[1].legend(fontsize=8)
    fig.savefig(FIGURE_DIR / "spectral_1064_modes_v2.png", bbox_inches="tight")
    plt.show()

## Repeatability is not accuracy

Files are grouped only when their physical settings match and they are time-contiguous. High-count warning files are excluded from the repeatability summary. A standard deviation within a sequence describes acquisition repeatability; it does not include wavelength-calibration error, slit-position error, instrument drift outside the sequence, or spectral-response correction.

In [ ]:
display(repeatability[repeatability.n.gt(1)].round(6))

## Picosecond-mode spectral bandwidth and transform limits

The picosecond-mode FWHM is well sampled: approximately 8.49–9.61 nm, depending on grating and acquisition. For an optical frequency FWHM \(\Delta\nu\), ideal transform-limited intensity durations are

\[
\Delta t_\mathrm{TL}=\frac{0.441}{\Delta\nu}\quad\text{(Gaussian)},\qquad
\Delta t_\mathrm{TL}=\frac{0.315}{\Delta\nu}\quad\text{(sech}^2\text{)}.
\]

These values are lower bounds. The actual pulse can be longer because of chirp or arbitrary spectral phase, and the spectral response of the spectrometer/detector was not corrected.

In [ ]:
display(ps_limits[[
    "grating_g_mm", "slit_um_filename", "peak_wavelength_nm", "centroid_wavelength_nm",
    "fwhm_nm", "fwhm_frequency_ghz", "points_per_fwhm",
    "transform_limit_gaussian_ps", "transform_limit_sech2_ps",
]].round(6))

print(
    f'Gaussian transform-limit range: {ps_limits.transform_limit_gaussian_ps.min():.3f}–'
    f'{ps_limits.transform_limit_gaussian_ps.max():.3f} ps.'
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for grating, d in ps_limits.groupby("grating_g_mm"):
    ax.scatter(d.fwhm_nm, d.transform_limit_gaussian_ps, label=f"{int(grating)} gr/mm")
ax.set_xlabel("Measured picosecond-mode spectral FWHM [nm]")
ax.set_ylabel("Ideal Gaussian transform limit [ps]")
ax.set_title("Transform-limit lower bound, not measured pulse duration")
ax.legend()
fig.savefig(FIGURE_DIR / "spectral_transform_limits_v2.png", bbox_inches="tight")
plt.show()

## Spectral confidence verdict

| Quantity | Verdict | Interpretation |
|---|---|---|
| 532 nm center wavelength | **Supported with calibration caveat** | Visible Hg calibration is regionally relevant, but residuals/absolute uncertainty were not saved. |
| 532 nm intrinsic linewidth | **Not supported** | All measured peaks have fewer than 2.2 samples across FWHM and instrument response is not removed. |
| 1064 CW/65 ns center near 1059–1061 nm | **Physically plausible, absolute value unvalidated** | It lies within 1064 ± 10 nm, but visible Hg calibration was extrapolated to the near-IR. |
| 1064 narrow intrinsic linewidth | **Not supported** | Slit dependence and undersampling/instrument response dominate. |
| 65 ns high-count broad spectra | **Not reportable as ordinary repeats** | Distinct acquisition state and ≥85% full-scale QC warning. |
| Picosecond-mode bandwidth | **Supported with caveat** | 8.49–9.61 nm and well sampled; system spectral response is uncorrected. |
| Transform-limited duration | **Supported as a lower bound only** | Gaussian 0.170–0.192 ps; actual pulse needs phase-sensitive or autocorrelation/FROG measurement. |